# 15 - Word2Vec with Negative Sampling

Goal: Implement Word2Vec efficiently using negative sampling

Run with: conda activate tfenv

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import plotly.express as px
import pandas as pd
import time

print(f"TensorFlow version: {tf.__version__}")

I0000 00:00:1779039779.609148   27447 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779039779.762466   27447 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1779039779.672209   27447 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://ipywidgets.readthedocs.io/en/stable/user_install.html for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


TensorFlow version: 2.21.0


In [ ]:
# Load dataset from HuggingFace
print("Loading gaianet/london dataset...")
from datasets import load_dataset

ds = load_dataset("gaianet/london", split="train")

# Extract text
print(f"Dataset size: {len(ds)}")
texts = [row['text'] if 'text' in row else row.get('content', '') for row in ds][:10000]
text = ' '.join(texts[:5000])
print(f"Total chars: {len(text)}")

text = """
El mercado de tecnologías de la información ha experimentado un crecimiento sin precedentes en los últimos años. Las empresas invierten cada vez más en soluciones digitales para mejorar su eficiencia y competitividad. La inteligencia artificial se ha convertido en una herramienta fundamental para automatizar procesos y tomar decisiones basadas en datos. Los profesionales del sector tecnológico buscan constantemente actualizar sus habilidades para mantenerse al día con las últimas tendencias. El aprendizaje automático permite a las máquinas mejorar su rendimiento a través de la experiencia sin ser programadas explícitamente. Las redes neuronales profundas han revolucionado campos como el reconocimiento de voz y la visión por computadora. El procesamiento del lenguaje natural ha avanzado significativamente permitiendo a las máquinas entender y generar texto de manera más natural. Los modelos de lenguaje grandes como GPT han demostrado capacidades sorprendentes en múltiples tareas. La seguridad cibernética se ha vuelto crítica ya que las amenazas digitales evolucionan constantemente. Las empresas necesitan proteger sus datos y sistemas de ataques maliciosos. La computación en la nube ofrece flexibilidad y escalabilidad para las organizaciones. Los servicios web permiten a las aplicaciones comunicarse e intercambiar información de manera eficiente. Las bases de datos almacenan información estructurada que puede ser consultada y analizada. Los desarrolladores utilizan lenguajes de programación como Python Java y JavaScript para crear soluciones innovadoras. Los frameworks facilitan el desarrollo de aplicaciones web y móviles con interfaces modernas. La metodología ágil promueve la colaboración y la entrega iterativa de valor al cliente. Los equipos de desarrollo trabajan de manera colaborativa utilizando herramientas de control de versiones. La documentación técnica es esencial para mantener y evolucionar los sistemas de información. Las pruebas automatizadas garantizan la calidad del software y reducen errores en producción. Los contenedores y la orquestación permiten desplegar aplicaciones de manera consistente. Los microservicios facilitan la evolución independiente de componentes del sistema. Las API RESTful permiten la integración entre diferentes servicios y plataformas. El análisis de datos proporciona insights valiosos para la toma de decisiones empresariales. Los dashboards y visualizaciones facilitan la comprensión de información compleja. El machine learning se aplica en Recomendaciones detección de fraudes y optimización de procesos. Los modelos predictivos permiten anticipar tendencias y comportamientos futuros. La ética en inteligencia artificial es un tema cada vez más importante para las organizaciones. La regulación de datos como GDPR impacto la manera en que las empresas manejan información personal. La formación continua es esencial para mantenerse competitivo en el mercado laboral tecnológico. Los bootcamps y cursos online ofrecen alternativas accesibles para aprender programación. La comunidad de código abierto contribuye significativamente al ecosistema tecnológico. Los estándares y mejores prácticas ayudan a construir software más robusto y mantenible. La automatización de procesos reduce errores y mejora la eficiencia operativa. La monitorización y logging son esenciales para detectar problemas en sistemas de producción. La escalabilidad permite manejar crecimiento en usuarios y carga de trabajo. La resiliencia asegura la disponibilidad de servicios ante fallos y contingencias.
"""

words = text.lower().split()
words = [w.strip('.,;:!?()[]"\'-0123456789') for w in words]
words = [w for w in words if len(w) > 2]

print(f"Total words: {len(words)}")
print(f"Sample: {words[:20]}")

Total words: 2721
Sample: ['mercado', 'tecnologías', 'información', 'experimentado', 'crecimiento', 'sin', 'precedentes', 'los', 'últimos', 'años', 'las', 'empresas', 'invierten', 'cada', 'vez', 'más', 'soluciones', 'digitales', 'para', 'mejorar']


In [ ]:
# Build vocabulary
from collections import Counter
word_counts = Counter(words)
vocab = [w for w, c in word_counts.items() if c >= 3]

text_vocab = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in text_vocab.items()}
text_vocab_size = len(text_vocab)
text_embed_dim = 64

print(f"Vocabulary: {text_vocab_size} words")

Vocabulary: 178 words


In [ ]:
# Create training pairs (Skip-gram)
def create_pairs(words, window=2):
    pairs = []
    for i, word in enumerate(words):
        if word not in text_vocab:
            continue
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if i != j and words[j] in text_vocab:
                pairs.append((text_vocab[word], text_vocab[words[j]]))
    return pairs

pairs = create_pairs(words)
train_words = np.array([p[0] for p in pairs], dtype=np.int32)
train_context = np.array([p[1] for p in pairs], dtype=np.int32)
print(f"Training pairs: {len(pairs)}")

Training pairs: 2482


In [ ]:
# Generate negative samples
NEGATIVE_SAMPLING = 5

all_indices = np.arange(text_vocab_size)

np.random.seed(42)
negative_samples = []

for target_idx in train_context:
    neg_indices = np.random.choice(all_indices, size=NEGATIVE_SAMPLING, replace=False)
    for neg_idx in neg_indices:
        negative_samples.append((target_idx, neg_idx))

neg_words = np.array([p[0] for p in negative_samples], dtype=np.int32)
neg_context = np.array([p[1] for p in negative_samples], dtype=np.int32)
neg_labels = np.zeros(len(negative_samples), dtype=np.float32)

print(f"NEGATIVE_SAMPLING = {NEGATIVE_SAMPLING}")
print(f"Generated {len(negative_samples)} negative samples")

NEGATIVE_SAMPLING = 5
Generated 12410 negative samples


In [ ]:
# Combine positive and negative samples
pos_labels = np.ones(len(train_words), dtype=np.float32)

all_words = np.concatenate([train_words, neg_words])
all_context = np.concatenate([train_context, neg_context])
all_labels = np.concatenate([pos_labels, neg_labels])

print(f"Total training samples: {len(all_words)}")
print(f"  Positive (1): {len(pos_labels)}")
print(f"  Negative (0): {len(neg_labels)}")

Total training samples: 16892


In [ ]:
# Word2Vec with Negative Sampling - Custom Model
class Word2VecNS(keras.Model):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.target_embedding = layers.Embedding(
            vocab_size, embed_dim, name='embedding_target'
        )
        self.context_embedding = layers.Embedding(
            vocab_size, embed_dim, name='embedding_context'
        )
        self.dense = layers.Dense(1, activation='sigmoid')

    def call(self, target, context):
        target_emb = self.target_embedding(target)
        context_emb = self.context_embedding(context)
        dot_product = tf.reduce_sum(target_emb * context_emb, axis=-1)
        return self.dense(dot_product)

model = Word2VecNS(text_vocab_size, text_embed_dim)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Word2Vec with Negative Sampling model ready!")
model.summary()

Could not find a functional way to connect these layers. Building model with tf.function.


Word2Vec with Negative Sampling model ready!


E0000 00:00:1779039788.631132   27447 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading symbol. GPU will not be used.: Error loading symbol. GPU will not be used.


Total params: 22848 (89.25 KB)
  • embedding_target: 11392 (44.50 KB)
  • embedding_context: 11392 (44.50 KB)


In [ ]:
# Training with negative sampling
train_ds = tf.data.Dataset.from_tensor_slices((
    (all_words, all_context),
    all_labels
))
train_ds = train_ds.shuffle(buffer_size=len(all_words)).batch(256).prefetch(tf.data.AUTOTUNE)

print("Training Word2Vec with Negative Sampling...")
start = time.time()

history = model.fit(
    train_ds,
    epochs=20,
    verbose=1
)

print(f"Training time: {time.time() - start:.2f}s")

Training Word2Vec with Negative Sampling...
Epoch 1/20
55/55 ━━━━━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - accuracy: 0.6855 - loss: 0.6029
Epoch 2/20
55/55 ━━━━━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.8569 - loss: 0.3629
Epoch 3/20
55/55 ━━━━━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.8910 - loss: 0.2860
Epoch 4/20
55/55 ━━━━━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - loss: 0.2281
Epoch 5/20
05/55 ━━━━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - loss: 0.1727
Epoch 5/20: 55/55 ━━━━━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.9316 - loss: 0.1727
Epoch 6/20
55/55 ━━━━━━━━━━━━━━━━━crumbs━━ 2s 32ms/step - accuracy: 0.9439 - loss: 0.1612
Epoch 7/20
55/55 ━━━━━━━━━━━━━━━━━━━━━━━━ 2s era 2s 31ms/step - accuracy: 0.9524 - loss: 0.0
Epoch 7/20: 55/55 ━━━━━━━━━━━━━━━━━━━━━━━━ 2s 32ms 式 step - accuracy: 0.9524 - loss: 0.1303
Epoch 8/20
55/55 ━━━━━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.9575 - loss: 0.1139
Epoch 9/20
55/55 55/55 ━━━━━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.9605 - loss: 0.0994
Epo

In [ ]:
# Get embeddings (use target embedding)
target_embeddings = model.target_embedding.get_weights()[0]
context_embeddings = model.context_embedding.get_weights()[0]

final_embeddings = (target_embeddings + context_embeddings) / 2

print(f"Target embeddings shape: {target_embeddings.shape}")
print(f"Context embeddings shape: {context_embeddings.shape}")

Target embeddings shape: (178, 64)
Context embeddings shape: (178, 64)


In [ ]:
# Find similar words
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

def find_similar(word, top_n=5):
    if word not in text_vocab:
        return []
    word_idx = text_vocab[word]
    word_emb = final_embeddings[word_idx]
    
    similarities = []
    for w, idx in text_vocab.items():
        if w != word:
            sim = cosine_similarity(word_emb, final_embeddings[idx])
            similarities.append((w, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

print("=== Similar Words (Negative Sampling) ===")
test_words = ['tecnología', 'empresas', 'datos', 'desarrollo', 'aprendizaje', 'software']
for word in test_words:
    if word in text_vocab:
        similar = find_similar(word)
        print(f"'{word}' -> {[(w, f'{s:.3f}') for w, s in similar]}")

=== Similar Words (Negative Sampling) ===
'tecnología' -> [('sistemas', '0.960'), ('inteligencia', '0.957'), ('datos', '0.948'), ('modelos', '0.946'), ('modelo', '0.943')]
'empresas' -> [('empleo', '0.939'), ('desarrollo', '0.923'), ('estrategias', '0.912'), ('tecnológicas', '0.907'), ('mejores', '0.903')]
'datos' -> [('basadas', '0.957'), ('inteligencia', '0.953'), ('herramienta', '0.946'), ('procesos', '0.943'), ('información', '0.941')]
'desarrollo' -> [('empleo', '0.937'), ('mejores', '0.920'), ('estrategias', '0.918'), ('sistemas', '0.915'), ('estrategia', '0.912')]
'aprendizaje' -> [('máquinas', '0.963'), ('inteligencia', '0.944'), ('artificial', '0.934'), ('servicios', '0.923'), ('basadas', '0.920')]
'software' -> [('sistemas', '0.918'), ('desarrollo', '0.904'), ('pruebas', '0.901'), ('aplicaciones', 'pruebas', '0.900'), ('mantener', '0.894')]


In [ ]:
# Test analogies
def analogy(word_a, word_b, word_c):
    if word_a not in text_vocab or word_b not in text_vocab or word_c not in text_vocab:
        return None
    
    vec = final_embeddings[text_vocab[word_b]] - final_embeddings[text_vocab[word_a]] + final_embeddings[text_vocab[word_c]]
    
    similarities = []
    for w, idx in text_vocab.items():
        if w not in [word_a, word_b, word_c]:
            sim = cosine_similarity(vec, final_embeddings[idx])
            similarities.append((w, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[0]

print("=== Analogies ===")
result = analogy('inteligencia', 'artificial', 'aprendizaje')
if result:
    print(f"'inteligencia' - 'artificial' + 'aprendizaje' = '{result[0]}' ({result[1]:.3f})")

result = analogy('empresas', 'tecnología', 'datos')
if result:
    print(f"'empresas' - 'tecnología' + 'datos' = '{result[0]}' ({result[1]:.3f})")

=== Analogies ===
'inteligencia' - 'artificial' + 'aprendizaje' = 'automático' (0.961)
'empresas' - 'tecnología' + 'datos' = 'información' (0.965)


In [ ]:
# Visualize in 3D with PCA
from sklearn.decomposition import PCA

top_words = [w for w, c in word_counts.most_common(200) if w in text_vocab]
top_indices = [text_vocab[w] for w in top_words]
top_embeddings = final_embeddings[top_indices]

pca = PCA(n_components=3)
embeddings_3d = pca.fit_transform(top_embeddings)

df = pd.DataFrame(embeddings_3d, columns=['x', 'y', 'z'])
df['word'] = top_words

fig = px.scatter_3d(df, x='x', y='y', z='z', text='word',
                 title='Word2Vec with Negative Sampling - Embeddings (PCA 3D)')
fig.update_traces(marker=dict(size=6), textposition='top center')
fig.update_layout(height=600, width=800)
fig.show()

In [ ]:
# Summary
print("=== Summary ===")
print(f"Vocab size: {text_vocab_size}")
print(f"Embedding dim: {text_embed_dim}")
print(f"Positive pairs: {len(train_words)}")
print(f"Negative samples: {len(neg_labels)}")
print(f"Total training: {len(all_words)}")
print(f"Negative sampling ratio: {NEGATIVE_SAMPLING}:1")
print("\nModel uses:")
print("  - Two embeddings: target and context")
print("  - Dot product + sigmoid (binary classification)")
print("  - Much faster than full softmax!")